# Association-rule performance visualization

Explore monthly confidence, lift, and rule-count trends for selected products.


## Imports


In [ ]:
import os
import pyodbc
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy.engine import URL
from sqlalchemy import create_engine, text
from sklearn.preprocessing import MinMaxScaler

## Load association rules from the SQL Server source


In [ ]:
# Load environment variables from the project root when available.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

datasets_dir = project_root / "datasets"
env_path = project_root / ".env"
if env_path.exists():
    load_dotenv(env_path)
else:
    load_dotenv()

dw_user = (os.getenv("DATAWAREHOUSE_USER") or "").strip()
dw_password = (os.getenv("DATAWAREHOUSE_PASSWORD") or "").strip()
dw_host = (os.getenv("DATAWAREHOUSE_HOST") or "").strip()
dw_database = (os.getenv("DATAWAREHOUSE_DATABASE") or "").strip()

if not all([dw_user, dw_password, dw_host, dw_database]):
    raise ValueError("Missing one or more DATAWAREHOUSE_* environment variables.")

drivers = pyodbc.drivers()
print("Available ODBC drivers:", drivers)

preferred_driver_names = [
    (os.getenv("SQLSERVER_DRIVER") or "").strip(),
    "ODBC Driver 18 for SQL Server",
    "ODBC Driver 17 for SQL Server",
    "SQL Server",
]

driver = next(
    (
        candidate
        for candidate in preferred_driver_names
        if candidate and any(candidate.lower() in d.lower() for d in drivers)
    ),
    None,
)

if not driver:
    raise RuntimeError(
        f"No supported SQL Server ODBC driver found. Available drivers: {drivers}. "
        "Install Microsoft ODBC Driver 18 for SQL Server and verify the 64-bit ODBC Administrator."
    )

print(f"Using SQL Server driver: {driver}")

connection_string = URL.create(
    drivername="mssql+pyodbc",
    username=dw_user,
    password=dw_password,
    host=dw_host,
    database=dw_database,
    query={
        "driver": driver,
        "Encrypt": "yes",
        "TrustServerCertificate": "yes",
        "LoginTimeout": "30",
    },
)

engine = create_engine(connection_string, pool_pre_ping=True, future=True)

with engine.connect() as conn:
    print("Database connection OK:", conn.execute(text("SELECT 1")).scalar())

query = '''
    SELECT * FROM FactProductAssociationsStore WHERE store_code = '038'
    '''

df = pd.read_sql(query, engine)

## Filter candidate rules


In [ ]:
# Create a rule label column like "Aqua 600ml → Fruit Cut"
df['rule'] = df['antecedents_names'] + " → " + df['consequents_names']

required_items = {"CHICKEN BREAST BONELESS KG", "WORTEL MEDAN KG"}


filtered_rules = df[df['antecedents_names'].apply(
    lambda x: required_items.issubset({item.strip() for item in x.split(",")})
)]
filtered_rules.head(5)

rules_sort = filtered_rules.sort_values(by='confidence', ascending=False) 

rules_sort = rules_sort.head(5)

rules_sort 

## Calculate monthly item metrics


In [ ]:
item = "CHICKEN BREAST BONELESS KG"

confidence_trend = (
    df[
        df['antecedents_names'].str.contains(item) | df['consequents_names'].str.contains(item)
    ]
    .groupby('month_year')['confidence']
    .mean()   # could also use .max() or .median()
    .reset_index()
    .sort_values('month_year')
)

confidence_trend

In [ ]:
lift_trend = (
    df[
        df['antecedents_names'].str.contains(item) | df['consequents_names'].str.contains(item)
    ]
    .groupby('month_year')['lift']
    .mean()   # could also use .max() or .median()
    .reset_index()
    .sort_values('month_year')
)

lift_trend

In [ ]:
rule_count = (
    df[
        df['antecedents_names'].str.contains(item) | df['consequents_names'].str.contains(item)
    ]
    .groupby('month_year')
    .size()
    .reset_index(name='rule_count')
    .sort_values('month_year')
)

rule_count

In [ ]:
item_performance = (
    df[
        df['antecedents_names'].str.contains(item) | df['consequents_names'].str.contains(item)
    ]
    .groupby('month_year')
    .agg(
        avg_confidence=('confidence', 'mean'),
        avg_lift=('lift', 'mean'),
        rule_count=('confidence', 'size')
    )
    .reset_index()
    .sort_values('month_year')
)

item_performance

## Plot raw performance trends


In [ ]:

plt.figure(figsize=(10,6))
plt.plot(item_performance['month_year'], item_performance['avg_confidence'], marker='o', label='Avg Confidence')
plt.plot(item_performance['month_year'], item_performance['avg_lift'], marker='s', label='Avg Lift')

plt.title(f"Performance Trend of {item}")
plt.xlabel("Month-Year")
plt.ylabel("Value")
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

## Plot normalized confidence and lift


In [ ]:
scaler = MinMaxScaler()
scaled = scaler.fit_transform(item_performance[['avg_confidence','avg_lift']])
item_performance[['conf_scaled','lift_scaled']] = scaled

plt.figure(figsize=(10,6))
plt.plot(item_performance['month_year'], item_performance['conf_scaled'], marker='o', label='Confidence (scaled)')
plt.plot(item_performance['month_year'], item_performance['lift_scaled'], marker='s', label='Lift (scaled)')
plt.legend()
plt.xticks(rotation=45)
plt.title(f"Normalized Trends for {item}")
plt.show()
